# v0.3.10: Excitatory-Inhibitory EEG, MEG, and EMM Proxy Bundle

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v0310_eeg_meg_emm_proxy_bundle.ipynb)

**Duration:** 15–20 minutes | **Difficulty:** Intermediate | **v0.3.10+**

## 1. Learning Objectives

By the end of this tutorial, you will:

- Construct a balanced population (100 neurons) via the fluent Configuration API
- Generate completely separate EEG-proxy, MEG-proxy, and EMM-proxy operators
- Extract distinct multi-channel scalp potentials, magnetometer readings, and metabolic proxies
- Plot separate visualization panels for each macroscopic sensor path
- Verify JSON metadata reports and export a structured validation manifest

## 2. Biological/Computational Question

**Question:** How can we project microscopic laminar source currents onto distinct macroscopic sensors (EEG-proxy, MEG-proxy, and metabolic EMM-proxy) with independent signal paths?

**Context:** 
EEG projects potential fields through a volume conductor, whereas MEG registers magnetic fields induced by intracellular current dipoles. Electromagnetic Metabolism Estimates (EMM) summarize energetic consumption. These three macroscopic streams offer complementary views of the same underlying laminar population activity.

## 3. Mathematical Glossary Flow

### 1. EEG-Proxy Sensor Equation

The EEG-proxy scalp potential $Y_{\text{EEG}, s}(t)$ at sensor $s$ maps linearly via a leadfield projection:

$$Y_{\text{EEG}, s}(t) = \sum_{n=1}^{N} W^{\text{EEG}}_{sn} S_n(t)$$

where $W^{\text{EEG}}_{sn}$ is the leadfield weight from source $n$ to sensor $s$, and $S_n(t)$ is the source current.

### 2. MEG-Proxy Magnetometer Equation

The MEG-proxy magnetic field $Y_{\text{MEG}, m}(t)$ at sensor $m$ projects based on oriented current dipoles:

$$Y_{\text{MEG}, m}(t) = \sum_{n=1}^{N} W^{\text{MEG}}_{mn} S_n(t)$$

where $W^{\text{MEG}}_{mn}$ is the magnetic coupling coefficient.

### 3. EMM-Proxy Metabolism Equation

The metabolic activity proxy $Y_{\text{EMM}}(t)$ tracks total electrophysiological field activity cost over time:

$$Y_{\text{EMM}}(t) = \frac{1}{N} \sum_{n=1}^{N} |S_n(t)|$$

In [ ]:
import jaxfne as jtfne
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import json

print(f"jaxfne version: {jtfne.__version__}")

## 4. Fluent Configuration Setup

Configure a 100-neuron population with separate multimodal sensors.

In [ ]:
cfg = (jtfne.Configuration()
    .runtime(seed=42, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
    .column(name="L2/3_column", layers=["L2/3"], n=100)
    .cell_types({"E": 0.75, "I": 0.25})
    .connectivity()
    .probes(["spk", "vm", "source", "lfp_proxy", "csd_proxy", "eeg_proxy", "meg_proxy", "emm_proxy"]))

print("Configuration complete.")

## 5. Construction and Simulation

Build the model and execute the vectorized JAX simulation.

In [ ]:
model = jtfne.construct(cfg)
signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=42)

lfp_data = signals.field.lfp_proxy if hasattr(signals, 'field') and signals.field is not None else jnp.zeros((10000, 4))
print(f"LFP data shape: {lfp_data.shape}")

## 6. EEG-Proxy Sensor Projections

Compute separate scalp-channel EEG potential readouts.

In [ ]:
eeg_readout = jtfne.fields.eeg_proxy_probe(lfp_data, leadfield_status="toy_or_declared_proxy", n_sensors=4)
print("EEG Report Metadata:")
print(json.dumps(eeg_readout.report, indent=2))

plt.figure(figsize=(10, 3))
plt.plot(eeg_readout.data[:, :4])
plt.title("EEG Scalp Channel Projections (Separate Path)")
plt.xlabel("Timesteps")
plt.ylabel("Amplitude (Arbitrary Proxy Units)")
plt.tight_layout()
plt.savefig("suite_no4_eeg_readouts.png")
plt.close()

## 7. MEG-Proxy Magnetometer Projections

Compute separate current-orientation magnetic fields.

In [ ]:
meg_readout = jtfne.fields.meg_proxy_probe(lfp_data, leadfield_status="toy_or_declared_proxy", orientation_convention="declared", n_sensors=4)
print("MEG Report Metadata:")
print(json.dumps(meg_readout.report, indent=2))

plt.figure(figsize=(10, 3))
plt.plot(meg_readout.data[:, :4])
plt.title("MEG Magnetometer Projections (Separate Path)")
plt.xlabel("Timesteps")
plt.ylabel("Amplitude (Arbitrary Proxy Units)")
plt.tight_layout()
plt.savefig("suite_no4_meg_readouts.png")
plt.close()

## 8. Metabolic EMM-Proxy Tracks

Compute separate electromagnetic metabolism estimate proxy timelines.

In [ ]:
emm_data = jnp.mean(jnp.abs(lfp_data), axis=1)
emm_readout = jtfne.fields.emm_proxy_probe(emm_data, method="normalized_activity_field_source_cost_proxy")
print("EMM Report Metadata:")
print(json.dumps(emm_readout.report, indent=2))

plt.figure(figsize=(10, 3))
plt.plot(emm_readout.data)
plt.title("EMM Activity Cost Trajectory (Separate Path)")


*Configure simulation parameters and environment state.*


In [ ]:
plt.xlabel("Timesteps")
plt.ylabel("Energy Proxy Cost")
plt.tight_layout()
plt.savefig("suite_no4_emm_readouts.png")
plt.close()

## 9. Validation and Manifest Export

Export execution receipt confirming separate operators.

In [ ]:
receipt = {
    "eeg_finite": bool(jnp.all(jnp.isfinite(eeg_readout.data))),
    "meg_finite": bool(jnp.all(jnp.isfinite(meg_readout.data))),
    "emm_finite": bool(jnp.all(jnp.isfinite(emm_readout.data))),
    "eeg_shape": list(eeg_readout.data.shape),
    "meg_shape": list(meg_readout.data.shape),
    "emm_shape": list(emm_readout.data.shape),
    "non_negotiable_boundaries": {
        "physical_amplitude_claim_allowed": False,
        "field_solver_status": "laminar_proxy_no_pde",
        "science_mode": "truth_safe_unverified"
    }
}

with open("suite_no4_v0310_execution_receipt.json", "w") as f:
    json.dump(receipt, f, indent=2)

print("Receipt exported.")

## 10. Scope Boundaries

### What This Tutorial Is
✓ A computational scaffold for learning multi-sensor proxy mappings  
✓ A demonstration of configured separate signal streams  
✓ JAX-based, deterministic, and reproducible  

### What This Tutorial Is NOT
✗ A biophysically validated dataset  
✗ A calibrated physical field solver  

### Scope Metadata
```json
{
  "scope_status": "computational_scaffold",
  "calibration_status": "uncalibrated_phenomenological",
  "readout_status": "proxy_scale",
  "field_mode": "proxy_convolution_no_pde",
  "physical_amplitude_claim_allowed": false
}
```